# More Applied Econometric Examples
This notebook demonstrates three additional applied examples, comparing `stats-transformer` models against native `statsmodels` baselines to ensure exact numerical replication.

# 1. Longley Macroeconomic Data
The Longley dataset is a classic macroeconomic dataset known for high multicollinearity. We will fit a standard OLS regression.

In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from stats_transformer.models.regression.regression import RegressionModel
from stats_transformer.models.regression.robust_ols import RobustOLSModel

print("Fetching Longley Data...")
longley_raw = sm.datasets.longley.load_pandas().data
display(longley_raw.head())

Fetching Longley Data...


,TOTEMP,GNPDEFL,GNP,UNEMP,ARMED,POP,YEAR
0,60323.0,83.0,234289.0,2356.0,1590.0,107608.0,1947.0
1,61122.0,88.5,259426.0,2325.0,1456.0,108632.0,1948.0
2,60171.0,88.2,258054.0,3682.0,1616.0,109773.0,1949.0
3,61187.0,89.5,284599.0,3351.0,1650.0,110929.0,1950.0
4,63221.0,96.2,328975.0,2099.0,3099.0,112075.0,1951.0


## 1.1 Baseline: Native Statsmodels
Fitting OLS with native statsmodels.

In [2]:
y1 = longley_raw['TOTEMP']
X1 = longley_raw[['GNPDEFL', 'GNP', 'UNEMP', 'ARMED', 'POP', 'YEAR']]
X1 = sm.add_constant(X1)
sm_model1 = sm.OLS(y1, X1).fit()
print(sm_model1.summary().tables[1])

                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const      -3.482e+06    8.9e+05     -3.911      0.004    -5.5e+06   -1.47e+06
GNPDEFL       15.0619     84.915      0.177      0.863    -177.029     207.153
GNP           -0.0358      0.033     -1.070      0.313      -0.112       0.040
UNEMP         -2.0202      0.488     -4.136      0.003      -3.125      -0.915
ARMED         -1.0332      0.214     -4.822      0.001      -1.518      -0.549
POP           -0.0511      0.226     -0.226      0.826      -0.563       0.460
YEAR        1829.1515    455.478      4.016      0.003     798.788    2859.515


## 1.2 Stats-Transformer
Fitting with `RegressionModel`.

In [3]:
st_model1 = RegressionModel(
    target='TOTEMP',
    independent_variables=['GNPDEFL', 'GNP', 'UNEMP', 'ARMED', 'POP', 'YEAR']
)
st_model1.fit(longley_raw)
print(st_model1.get_summary())

# Sanity Check
sm_params1 = sm_model1.params.sort_index()
st_params1 = st_model1.model.params.sort_index()
assert np.allclose(sm_params1.values, st_params1.values), "Longley coefficients mismatch!"
print("✅ Sanity check passed for Longley dataset.")

                            OLS Regression Results                            
Dep. Variable:                 TOTEMP   R-squared:                       0.995
Model:                            OLS   Adj. R-squared:                  0.992
Method:                 Least Squares   F-statistic:                     330.3
Date:                Fri, 08 May 2026   Prob (F-statistic):           4.98e-10
Time:                        10:49:07   Log-Likelihood:                -109.62
No. Observations:                  16   AIC:                             233.2
Df Residuals:                       9   BIC:                             238.6
Df Model:                           6                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const      -3.482e+06    8.9e+05     -3.911      0.0

# 2. Stackloss Data (Robust Standard Errors)
The Stackloss dataset contains observations from a plant oxidation of ammonia to nitric acid. We'll use Robust OLS (HC3 standard errors).

In [4]:
print("Fetching Stackloss Data...")
stackloss_raw = sm.datasets.stackloss.load_pandas().data
display(stackloss_raw.head())

Fetching Stackloss Data...


,STACKLOSS,AIRFLOW,WATERTEMP,ACIDCONC
0,42.0,80.0,27.0,89.0
1,37.0,80.0,27.0,88.0
2,37.0,75.0,25.0,90.0
3,28.0,62.0,24.0,87.0
4,18.0,62.0,22.0,87.0


## 2.1 Baseline: Native Statsmodels
Fitting OLS with HC3 robust standard errors.

In [5]:
y2 = stackloss_raw['STACKLOSS']
X2 = stackloss_raw[['AIRFLOW', 'WATERTEMP', 'ACIDCONC']]
X2 = sm.add_constant(X2)
sm_model2 = sm.OLS(y2, X2).fit(cov_type='HC3')
print(sm_model2.summary().tables[1])

                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const        -39.9197      9.001     -4.435      0.000     -57.562     -22.278
AIRFLOW        0.7156      0.213      3.353      0.001       0.297       1.134
WATERTEMP      1.2953      0.589      2.200      0.028       0.141       2.449
ACIDCONC      -0.1521      0.121     -1.262      0.207      -0.388       0.084


## 2.2 Stats-Transformer
Fitting with `RobustOLSModel` using HC3.

In [6]:
st_model2 = RobustOLSModel(
    target='STACKLOSS',
    independent_variables=['AIRFLOW', 'WATERTEMP', 'ACIDCONC'],
    cov_type='HC3'
)
st_model2.fit(stackloss_raw)
print(st_model2.get_summary())

# Sanity Check
sm_bse2 = sm_model2.bse.sort_index()
st_bse2 = st_model2.model.bse.sort_index()
assert np.allclose(sm_bse2.values, st_bse2.values), "Stackloss robust standard errors mismatch!"
print("✅ Sanity check passed for Stackloss dataset.")

                            OLS Regression Results                            
Dep. Variable:              STACKLOSS   R-squared:                       0.914
Model:                            OLS   Adj. R-squared:                  0.898
Method:                 Least Squares   F-statistic:                     47.18
Date:                Fri, 08 May 2026   Prob (F-statistic):           1.87e-08
Time:                        10:49:13   Log-Likelihood:                -52.288
No. Observations:                  21   AIC:                             112.6
Df Residuals:                      17   BIC:                             116.8
Df Model:                           3                                         
Covariance Type:                  HC3                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const        -39.9197      9.001     -4.435      0.0

# 3. Statecrime Data
Statewide crime data from the US. We will predict violent crime rates based on high school graduation rates and urban population percentages.

In [7]:
print("Fetching Statecrime Data...")
statecrime_raw = sm.datasets.statecrime.load_pandas().data
display(statecrime_raw.head())

Fetching Statecrime Data...


,violent,murder,hs_grad,poverty,single,white,urban
state,,,,,,,
Alabama,459.9,7.1,82.1,17.5,29.0,70.0,48.65
Alaska,632.6,3.2,91.4,9.0,25.5,68.3,44.46
Arizona,423.2,5.5,84.2,16.5,25.7,80.0,80.07
Arkansas,530.3,6.3,82.4,18.8,26.3,78.4,39.54
California,473.4,5.4,80.6,14.2,27.8,62.7,89.73


## 3.1 Baseline: Native Statsmodels
Fitting OLS with native statsmodels.

In [8]:
y3 = statecrime_raw['violent']
X3 = statecrime_raw[['hs_grad', 'urban']]
X3 = sm.add_constant(X3)
sm_model3 = sm.OLS(y3, X3).fit()
print(sm_model3.summary().tables[1])

                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const       1896.3133    681.001      2.785      0.008     527.070    3265.557
hs_grad      -19.7006      7.640     -2.579      0.013     -35.061      -4.340
urban          3.7370      1.241      3.013      0.004       1.243       6.231


## 3.2 Stats-Transformer
Fitting with `RegressionModel`.

In [9]:
st_model3 = RegressionModel(
    target='violent',
    independent_variables=['hs_grad', 'urban']
)
st_model3.fit(statecrime_raw)
print(st_model3.get_summary())

# Sanity Check
sm_params3 = sm_model3.params.sort_index()
st_params3 = st_model3.model.params.sort_index()
assert np.allclose(sm_params3.values, st_params3.values), "Statecrime coefficients mismatch!"
print("✅ Sanity check passed for Statecrime dataset.")

                            OLS Regression Results                            
Dep. Variable:                violent   R-squared:                       0.282
Model:                            OLS   Adj. R-squared:                  0.253
Method:                 Least Squares   F-statistic:                     9.449
Date:                Fri, 08 May 2026   Prob (F-statistic):           0.000347
Time:                        10:49:17   Log-Likelihood:                -335.61
No. Observations:                  51   AIC:                             677.2
Df Residuals:                      48   BIC:                             683.0
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const       1896.3133    681.001      2.785      0.0